In [1]:
import numpy as np
import pandas as pd
import time
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report

# Step 1: Generate Dataset
X, y = make_classification(
    n_samples=1000,
    n_features=20,
    n_informative=10,
    n_redundant=5,
    n_classes=2,
    weights=[0.9, 0.1],  # Imbalanced dataset
    random_state=42
)

print("Generating Imbalanced Placement Data:", X.shape)

# Step 2: Train-Test Split (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Step 3: Feature Scaling (Avoid Data Leakage)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)   # Fit only on training data
X_test = scaler.transform(X_test)         # Transform test data


# PHASE 2: BASELINE MODEL

print("\n--- BASELINE MODEL ---")

baseline_model = RandomForestClassifier(random_state=42)
baseline_model.fit(X_train, y_train)

y_pred = baseline_model.predict(X_test)

baseline_accuracy = accuracy_score(y_test, y_pred)
baseline_f1 = f1_score(y_test, y_pred)

print("Baseline Accuracy:", baseline_accuracy)
print("Baseline F1 Score:", baseline_f1)

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))


# GRID SEARCH SETUP

param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5, 10]
}

# GRID SEARCH (ACCURACY)

print("\n--- GRID SEARCH (ACCURACY) ---")

grid_acc = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    scoring='accuracy',
    cv=5,
    n_jobs=-1
)

grid_acc.fit(X_train, y_train)

print("Best Params (Accuracy):", grid_acc.best_params_)
print("Best Accuracy Score:", grid_acc.best_score_)

# GRID SEARCH (F1 SCORE)

print("\n--- GRID SEARCH (F1 SCORE) ---")

grid_f1 = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    scoring='f1',
    cv=5,
    n_jobs=-1
)

# Measure time for Grid Search
start_time = time.time()
grid_f1.fit(X_train, y_train)
grid_time = time.time() - start_time

print("Best Params (F1):", grid_f1.best_params_)
print("Best F1 Score:", grid_f1.best_score_)
print("Grid Search Time (seconds):", grid_time)


# RANDOMIZED SEARCH

print("\n--- RANDOMIZED SEARCH ---")

param_dist = {
    'n_estimators': np.arange(10, 500),
    'max_depth': [None] + list(np.arange(5, 50)),
    'min_samples_split': np.arange(2, 20)
}

random_search = RandomizedSearchCV(
    RandomForestClassifier(random_state=42),
    param_distributions=param_dist,
    n_iter=20,
    scoring='f1',
    cv=5,
    n_jobs=-1,
    random_state=42
)

start_time = time.time()
random_search.fit(X_train, y_train)
random_time = time.time() - start_time

print("Best Params (Random):", random_search.best_params_)
print("Best F1 Score (Random):", random_search.best_score_)
print("Random Search Time (seconds):", random_time)

# FINAL COMPARISON TABLE

print("\n--- FINAL COMPARISON ---")

results = pd.DataFrame({
    "Method": ["Grid Search", "Random Search"],
    "Time (seconds)": [grid_time, random_time],
    "Best F1 Score": [grid_f1.best_score_, random_search.best_score_]
})

print(results)

# FINAL MODEL EVALUATION

print("\n--- FINAL MODEL EVALUATION (BEST F1 MODEL) ---")

best_model = grid_f1.best_estimator_
final_predictions = best_model.predict(X_test)

print("Final Accuracy:", accuracy_score(y_test, final_predictions))
print("Final F1 Score:", f1_score(y_test, final_predictions))

print("\nFinal Classification Report:\n")
print(classification_report(y_test, final_predictions))

Generating Imbalanced Placement Data: (1000, 20)

--- BASELINE MODEL ---
Baseline Accuracy: 0.91
Baseline F1 Score: 0.3076923076923077

Classification Report:

              precision    recall  f1-score   support

           0       0.91      1.00      0.95       178
           1       1.00      0.18      0.31        22

    accuracy                           0.91       200
   macro avg       0.95      0.59      0.63       200
weighted avg       0.92      0.91      0.88       200


--- GRID SEARCH (ACCURACY) ---
Best Params (Accuracy): {'max_depth': None, 'min_samples_split': 2, 'n_estimators': 50}
Best Accuracy Score: 0.9199999999999999

--- GRID SEARCH (F1 SCORE) ---
Best Params (F1): {'max_depth': None, 'min_samples_split': 10, 'n_estimators': 50}
Best F1 Score: 0.4034632034632034
Grid Search Time (seconds): 32.207457065582275

--- RANDOMIZED SEARCH ---
Best Params (Random): {'n_estimators': np.int64(398), 'min_samples_split': np.int64(2), 'max_depth': np.int64(48)}
Best F1 Score (